# Credit Risk Project
## Phase 3 – Data Understanding

In [1]:
### 1. Import Libraries
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [2]:
pd.set_option('display.float_format', '{:,.2f}'.format)

### 3. Data quality Assesment 

### Observation

- Dataset contains 307,511 rows and 122 columns.
- Most categorical variables are correctly stored as `object`.
- Most numerical variables are correctly stored as `int64` or `float64`.
- Several columns that appear to be numerical are currently stored as `object`.
- These columns will be investigated further during the data quality assessment before any type conversion is performed.

The column `AMT_REQ_CREDIT_BUREAU_YEAR` contains 41,519 blank string values (`''`) representing missing data.

Because the column contains both numeric values and blank strings, Pandas inferred its data type as `object` instead of a numeric type.

This issue will be addressed during the Data Cleaning phase by replacing blank strings with `NaN` and converting the column to a numeric data type.

In [3]:
application_train_csv=pd.read_csv(r'C:\Users\HomePC\Desktop\home_credit_dataset\application_train.csv')

In [4]:
application_train_csv.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,"202,500.00","406,597.50","24,700.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,1.00
1,100003,0,Cash loans,F,N,N,0,"270,000.00","1,293,502.50","35,698.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.00","135,000.00","6,750.00",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00
3,100006,0,Cash loans,F,N,Y,0,"135,000.00","312,682.50","29,686.50",...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,"121,500.00","513,000.00","21,865.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00


In [5]:
application_train_csv.shape

(307511, 122)

In [6]:
application_train_csv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


In [7]:
application_train_csv.isnull().sum()

SK_ID_CURR                        0
TARGET                            0
NAME_CONTRACT_TYPE                0
CODE_GENDER                       0
FLAG_OWN_CAR                      0
                              ...  
AMT_REQ_CREDIT_BUREAU_DAY     41519
AMT_REQ_CREDIT_BUREAU_WEEK    41519
AMT_REQ_CREDIT_BUREAU_MON     41519
AMT_REQ_CREDIT_BUREAU_QRT     41519
AMT_REQ_CREDIT_BUREAU_YEAR    41519
Length: 122, dtype: int64

In [8]:
numeric_missing = (
    application_train_csv
    .select_dtypes(include=["int64", "float64"])
    .isna()
    .sum()
)

numeric_missing[numeric_missing>0]

AMT_ANNUITY                       12
AMT_GOODS_PRICE                  278
OWN_CAR_AGE                   202929
CNT_FAM_MEMBERS                    2
EXT_SOURCE_1                  173378
                               ...  
AMT_REQ_CREDIT_BUREAU_DAY      41519
AMT_REQ_CREDIT_BUREAU_WEEK     41519
AMT_REQ_CREDIT_BUREAU_MON      41519
AMT_REQ_CREDIT_BUREAU_QRT      41519
AMT_REQ_CREDIT_BUREAU_YEAR     41519
Length: 61, dtype: int64

In [9]:
numeric_missing[numeric_missing>0].reset_index().sort_values(0,ascending=False)

,index,0
25,COMMONAREA_MODE,214865
39,COMMONAREA_MEDI,214865
11,COMMONAREA_AVG,214865
33,NONLIVINGAPARTMENTS_MODE,213514
19,NONLIVINGAPARTMENTS_AVG,213514
...,...,...
5,EXT_SOURCE_2,660
1,AMT_GOODS_PRICE,278
0,AMT_ANNUITY,12
3,CNT_FAM_MEMBERS,2


In [10]:
missing_report = pd.DataFrame({
    "Missing Count": application_train_csv.isnull().sum()
})

missing_report["Missing %"] = (
    missing_report["Missing Count"] / len(application_train_csv) * 100
).round(2)

missing_report = (
    missing_report[missing_report["Missing Count"] > 0]
    .sort_values("Missing Count", ascending=False)
)

missing_report

,Missing Count,Missing %
COMMONAREA_MEDI,214865,69.87
COMMONAREA_AVG,214865,69.87
COMMONAREA_MODE,214865,69.87
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_MODE,213514,69.43
...,...,...
EXT_SOURCE_2,660,0.21
AMT_GOODS_PRICE,278,0.09
AMT_ANNUITY,12,0.00
CNT_FAM_MEMBERS,2,0.00


In [11]:
missing_report.shape

(67, 2)

In [12]:
application_train_csv.isnull().sum().gt(0).sum()

np.int64(67)

In [13]:
missing_report['Missing_Category']=pd.cut(missing_report['Missing %'],
                                          bins=[0,5,20,50,100],
                                          labels=['0-5%','5-20%','20-50%',
                                                  '50-100%'],include_lowest=True)

In [14]:
missing_report["Missing_Category"].value_counts().sort_index()

Missing_Category
0-5%       10
5-20%       7
20-50%      9
50-100%    41
Name: count, dtype: int64

## Observation: Missing Value Assessment

- The `application_train` dataset contains **122 features**, of which **67 features contain missing values** and **55 features have no missing values**.
- Missing values are represented as **NaN** in the original dataset.
- Missing values are not uniformly distributed across the dataset; they are concentrated in specific groups of features.
- Housing-related features such as `COMMONAREA_*`, `NONLIVINGAPARTMENTS_*`, `LANDAREA_*`, `YEARS_BUILD_*`, and similar variables have the highest percentage of missing values, with several columns containing approximately **70% missing data**.
- External credit score variables (`EXT_SOURCE_1`, `EXT_SOURCE_2`, and `EXT_SOURCE_3`) also contain missing values.
- Most financial and demographic variables contain relatively few missing values compared to housing-related features.
- The distribution of missing values across the dataset is as follows:
  - **10 columns** contain **0–5%** missing values.
  - **7 columns** contain **5–20%** missing values.
  - **9 columns** contain **20–50%** missing values.
  - **41 columns** contain **50–100%** missing values.
- Missing value treatment will be performed during the **Data Cleaning** phase after evaluating the business relevance and predictive importance of each feature.

In [15]:
duplicate_count = application_train_csv.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## Observation: Duplicate Record Assessment

- The `application_train` dataset contains **0 duplicate records**.
- Each record represents a unique loan application.
- No duplicate rows were identified in the dataset.
- Therefore, no duplicate records need to be removed during the Data Cleaning phase.

In [16]:
application_train_csv["SK_ID_CURR"].duplicated().sum()

np.int64(0)

### Primary Key Validation

- The `SK_ID_CURR` column contains **no duplicate values**.
- Each loan application is uniquely identified by `SK_ID_CURR`.
- This confirms that `SK_ID_CURR` can be treated as the unique identifier (Primary Key) for the `application_train` dataset.

In [17]:
categorical_columns=application_train_csv.select_dtypes(include='object').columns
print(categorical_columns.tolist())
print(len(categorical_columns))

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
16


In [18]:
for col in categorical_columns:
    print(f'{col}:{application_train_csv[col].nunique()}')

NAME_CONTRACT_TYPE:2
CODE_GENDER:3
FLAG_OWN_CAR:2
FLAG_OWN_REALTY:2
NAME_TYPE_SUITE:7
NAME_INCOME_TYPE:8
NAME_EDUCATION_TYPE:5
NAME_FAMILY_STATUS:6
NAME_HOUSING_TYPE:6
OCCUPATION_TYPE:18
WEEKDAY_APPR_PROCESS_START:7
ORGANIZATION_TYPE:58
FONDKAPREMONT_MODE:4
HOUSETYPE_MODE:3
WALLSMATERIAL_MODE:7
EMERGENCYSTATE_MODE:2


In [19]:
for col in categorical_columns:
    print(f'\n--- {col} ---')
    
    counts = application_train_csv[col].value_counts(dropna=False)
    percentages = application_train_csv[col].value_counts(
        normalize=True, dropna=False
    ).mul(100).round(2)
    
    result = pd.DataFrame({
        'Count': counts,
        'Percentage': percentages
    })
    
    print(result)


--- NAME_CONTRACT_TYPE ---
                     Count  Percentage
NAME_CONTRACT_TYPE                    
Cash loans          278232       90.48
Revolving loans      29279        9.52

--- CODE_GENDER ---
              Count  Percentage
CODE_GENDER                    
F            202448       65.83
M            105059       34.16
XNA               4        0.00

--- FLAG_OWN_CAR ---
               Count  Percentage
FLAG_OWN_CAR                    
N             202924       65.99
Y             104587       34.01

--- FLAG_OWN_REALTY ---
                  Count  Percentage
FLAG_OWN_REALTY                    
Y                213312       69.37
N                 94199       30.63

--- NAME_TYPE_SUITE ---
                  Count  Percentage
NAME_TYPE_SUITE                    
Unaccompanied    248526       80.82
Family            40149       13.06
Spouse, partner   11370        3.70
Children           3267        1.06
Other_B            1770        0.58
NaN                1292        0.42

### Categorical Data Quality Assessment – Observations

- Most categorical variables contain meaningful and well-defined categories with no major structural inconsistencies.
- `NAME_TYPE_SUITE` has low missingness (0.42%), while several housing-related variables have substantial missing values.
- `OCCUPATION_TYPE` contains 31.35% missing values and requires further investigation before deciding on an imputation strategy.
- `FONDKAPREMONT_MODE` (68.39%), `HOUSETYPE_MODE` (50.18%), `WALLSMATERIAL_MODE` (50.84%), and `EMERGENCYSTATE_MODE` (47.40%) contain very high proportions of missing values and require further assessment before modeling.
- `ORGANIZATION_TYPE` has relatively high cardinality with several low-frequency categories. The `XNA` category accounts for 18.01% of observations and should be investigated before treating it as missing.
- `CODE_GENDER` contains only 4 `XNA` records (approximately 0.00%), which is negligible but should be investigated as a potential special/missing value.
- `NAME_INCOME_TYPE` contains several very rare categories. These are not automatically considered errors and will be evaluated during feature engineering/modeling.
- No categorical variable is being dropped or modified at this stage. Cleaning and transformation decisions will be made after completing the overall data-quality assessment and understanding the business meaning of the variables.

In [20]:
numeric_columns = application_train_csv.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print(f'Number of numerical columns: {len(numeric_columns)}')

Number of numerical columns: 106


In [21]:
numeric_summary=application_train_csv[numeric_columns].describe().T
numeric_summary

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,"307,511.00","278,180.52","102,790.18","100,002.00","189,145.50","278,202.00","367,142.50","456,255.00"
TARGET,"307,511.00",0.08,0.27,0.00,0.00,0.00,0.00,1.00
CNT_CHILDREN,"307,511.00",0.42,0.72,0.00,0.00,0.00,1.00,19.00
AMT_INCOME_TOTAL,"307,511.00","168,797.92","237,123.15","25,650.00","112,500.00","147,150.00","202,500.00","117,000,000.00"
AMT_CREDIT,"307,511.00","599,026.00","402,490.78","45,000.00","270,000.00","513,531.00","808,650.00","4,050,000.00"
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,"265,992.00",0.01,0.11,0.00,0.00,0.00,0.00,9.00
AMT_REQ_CREDIT_BUREAU_WEEK,"265,992.00",0.03,0.20,0.00,0.00,0.00,0.00,8.00
AMT_REQ_CREDIT_BUREAU_MON,"265,992.00",0.27,0.92,0.00,0.00,0.00,0.00,27.00
AMT_REQ_CREDIT_BUREAU_QRT,"265,992.00",0.27,0.79,0.00,0.00,0.00,0.00,261.00


In [22]:
numeric_missing=pd.DataFrame({'missing_count':application_train_csv[numeric_columns]
                              .isnull().sum()})
numeric_missing['missing_percent']=(numeric_missing['missing_count']/len(application_train_csv)*100).round(2)
numeric_missing.sort_values(by='missing_percent',ascending=False)

,missing_count,missing_percent
COMMONAREA_MODE,214865,69.87
COMMONAREA_AVG,214865,69.87
COMMONAREA_MEDI,214865,69.87
NONLIVINGAPARTMENTS_AVG,213514,69.43
NONLIVINGAPARTMENTS_MODE,213514,69.43
...,...,...
DAYS_REGISTRATION,0,0.00
DAYS_EMPLOYED,0,0.00
DAYS_BIRTH,0,0.00
REGION_POPULATION_RELATIVE,0,0.00


### Numerical Data Quality Assessment – Missing Values

- The dataset contains 106 numerical features.
- Several numerical features contain substantial missing values, particularly property-related variables such as `COMMONAREA_*` and `NONLIVINGAPARTMENTS_*`, with approximately 69% missing observations.
- Several numerical features, including `DAYS_BIRTH`, `DAYS_EMPLOYED`, `DAYS_REGISTRATION`, `REGION_POPULATION_RELATIVE`, and `SK_ID_CURR`, have no missing values.
- Missingness is concentrated in groups of related property features, suggesting that the availability of certain property information varies across applicants.
- No numerical feature is being removed or imputed at this stage. Missing-value treatment will be decided after completing the overall data-quality assessment and understanding the feature distributions and business meaning.

In [23]:
application_train_csv[
    application_train_csv['CNT_CHILDREN'] >= 10
][['SK_ID_CURR', 'CNT_CHILDREN']]

,SK_ID_CURR,CNT_CHILDREN
34545,140032,11
80948,193853,12
132585,253779,10
155369,280108,19
171125,298322,12
176011,303956,14
183878,313127,14
186820,316580,10
265784,407877,19
267998,410527,14


In [24]:
application_train_csv[
    application_train_csv['AMT_INCOME_TOTAL'] > 10_000_000
][['SK_ID_CURR', 'AMT_INCOME_TOTAL']]

,SK_ID_CURR,AMT_INCOME_TOTAL
12840,114967,"117,000,000.00"
203693,336147,"18,000,090.00"
246858,385674,"13,500,000.00"


In [25]:
application_train_csv[
    application_train_csv['AMT_INCOME_TOTAL'] > 10_000_000
][[
    'SK_ID_CURR',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'NAME_INCOME_TYPE',
    'OCCUPATION_TYPE'
]]

,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,OCCUPATION_TYPE
12840,114967,"117,000,000.00","562,491.00","26,194.50",Working,Laborers
203693,336147,"18,000,090.00","675,000.00","69,295.50",Commercial associate,NaN
246858,385674,"13,500,000.00","1,400,503.50","130,945.50",Commercial associate,NaN


- `AMT_INCOME_TOTAL` shows substantial right-skewness, with a median income of 147,150 compared with a maximum of 117,000,000.
- Three observations have income values above 10 million, with the highest value being 117 million.
- These observations are extremely rare and are treated as potential extreme outliers rather than confirmed data errors.
- No records are removed or values modified at this stage. The impact of extreme values will be evaluated during EDA and model development.

In [26]:
negative_summary = pd.DataFrame({
    'Negative_Count': (application_train_csv[numeric_columns] < 0).sum()
})
negative_summary=negative_summary.sort_values(by='Negative_Count',ascending=False)
negative_summary

,Negative_Count
DAYS_BIRTH,307511
DAYS_ID_PUBLISH,307495
DAYS_REGISTRATION,307431
DAYS_LAST_PHONE_CHANGE,269838
DAYS_EMPLOYED,252135
...,...
YEARS_BUILD_AVG,0
YEARS_BEGINEXPLUATATION_AVG,0
BASEMENTAREA_AVG,0
APARTMENTS_AVG,0


### Negative Value Assessment

- Negative values were identified primarily in `DAYS_*` variables.
- Negative values in these variables are expected because they represent the number of days before the application date for events such as birth, employment, registration, ID publication, and phone changes.
- Therefore, negative values in these variables are not considered data-quality errors.
- `DAYS_EMPLOYED` requires additional investigation because it contains a special value representing an unusual/unknown employment status.
- No negative values were modified or removed at this stage.|

In [27]:
application_train_csv['DAYS_EMPLOYED'].value_counts(dropna=False).head(10)

DAYS_EMPLOYED
 365243    55374
-200         156
-224         152
-230         151
-199         151
-212         150
-384         143
-229         143
-231         140
-215         138
Name: count, dtype: int64

### DAYS_EMPLOYED – Special Value Assessment

- `DAYS_EMPLOYED` contains 55,374 observations with the value `365243`.
- The value `365243` is not a realistic employment duration and represents a special/unknown employment status in the dataset.
- The remaining negative values represent the number of days employed before the application date and are therefore valid according to the variable's representation.
- The special value `365243` will not be treated as a genuine numerical value during modeling.
- During preprocessing, this special value will be transformed appropriately while preserving the information that employment status was unavailable.

In [28]:
application_train_csv['DAYS_LAST_PHONE_CHANGE'].value_counts(dropna=False).head(10)

DAYS_LAST_PHONE_CHANGE
0.00       37672
-1.00       2812
-2.00       2318
-3.00       1763
-4.00       1285
-5.00        824
-6.00        537
-7.00        442
-8.00        278
-476.00      222
Name: count, dtype: int64

In [29]:
(application_train_csv[['DAYS_EMPLOYED','DAYS_LAST_PHONE_CHANGE']]==0).sum()

DAYS_EMPLOYED                 2
DAYS_LAST_PHONE_CHANGE    37672
dtype: int64

In [30]:
zero_summary=pd.DataFrame({'zero_count':(application_train_csv[numeric_columns]==0).sum()})
zero_summary['zero_percent']=  ((application_train_csv[numeric_columns] == 0).mean() * 100
    ).round(2)
zero_summary=zero_summary.sort_values(by='zero_percent',ascending=False)
zero_summary

,zero_count,zero_percent
FLAG_DOCUMENT_12,307509,100.00
FLAG_DOCUMENT_2,307498,100.00
FLAG_DOCUMENT_10,307504,100.00
FLAG_DOCUMENT_4,307486,99.99
FLAG_DOCUMENT_7,307452,99.98
...,...,...
REGION_RATING_CLIENT,0,0.00
EXT_SOURCE_1,0,0.00
EXT_SOURCE_2,0,0.00
EXT_SOURCE_3,0,0.00


### Numerical Data Quality Assessment – Zero Values

- Several numerical variables contain a very high proportion of zero values.
- The `FLAG_DOCUMENT_*` variables are particularly sparse, with some containing more than 99.9% zero values.
- Since these variables represent binary document-related indicators, the high proportion of zeros is not considered a data-quality error.
- However, variables with extremely low variation may provide limited predictive information and will be evaluated during feature selection/model development.
- Other numerical variables, including `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, and `SK_ID_CURR`, contain no zero values.
- No variables are removed solely based on their zero proportion at this stage.

In [31]:
constant_columns=[col for col in numeric_columns if application_train_csv[col]
                  .nunique(dropna=False)==1]
print(f'total number of constant columns are :{len(constant_columns)}')
print( constant_columns)

total number of constant columns are :0
[]


In [32]:
near_constant_columns = zero_summary[
    zero_summary['zero_percent'] >= 99
].sort_values(
    'zero_percent',
    ascending=False
)
near_constant_columns

,zero_count,zero_percent
FLAG_DOCUMENT_12,307509,100.00
FLAG_DOCUMENT_2,307498,100.00
FLAG_DOCUMENT_10,307504,100.00
FLAG_DOCUMENT_4,307486,99.99
FLAG_DOCUMENT_7,307452,99.98
FLAG_DOCUMENT_21,307408,99.97
FLAG_DOCUMENT_17,307429,99.97
FLAG_DOCUMENT_20,307355,99.95
FLAG_DOCUMENT_19,307328,99.94
FLAG_DOCUMENT_15,307139,99.88


### Near-Constant Numerical Features

- No numerical feature was found to be completely constant across all observations.
- However, 16 `FLAG_DOCUMENT_*` variables contain zero values in at least 99% of observations.
- These variables represent document-related indicators and their high proportion of zeros is not considered a data-quality error.
- Their extremely low variation may limit their predictive usefulness.
- These features will be evaluated during feature selection and model development rather than being removed during the current data-quality assessment.

In [33]:
application_train_csv.dtypes.value_counts()

float64    65
int64      41
object     16
Name: count, dtype: int64

In [34]:
object_columns = application_train_csv.select_dtypes(
    include='object'
).columns.tolist()

print(object_columns)
print(len(object_columns))

['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
16


## Data Quality Assessment – Summary

The `application_train` dataset contains **307,511 observations and 122 columns**. A systematic data-quality assessment was performed covering missing values, duplicate records, categorical distributions, numerical values, special values, zero values, constant and near-constant features, and data types.

### Key Findings

- **Duplicate records:** No duplicate rows were identified in the dataset.

- **Categorical variables:** The dataset contains **16 categorical (`object`) features**. Most categorical variables contain meaningful and well-defined categories.

- **Categorical missing values:** Missing values are present in several categorical variables. `FONDKAPREMONT_MODE` has the highest missingness at **68.39%**, followed by `WALLSMATERIAL_MODE` (**50.84%**), `HOUSETYPE_MODE` (**50.18%**), and `EMERGENCYSTATE_MODE` (**47.40%**).

- **`OCCUPATION_TYPE`:** Contains **31.35% missing values** and requires further assessment before deciding on an appropriate treatment strategy.

- **`ORGANIZATION_TYPE`:** Has relatively high cardinality and contains an `XNA` category representing **18.01%** of observations. This category requires further investigation before deciding whether it represents missing or unavailable information.

- **Rare categorical values:** Several categorical variables contain very low-frequency categories. These are not automatically considered errors and will be evaluated during feature engineering.

- **Numerical variables:** The dataset contains **106 numerical features**, consisting of 65 `float64` and 41 `int64` variables.

- **Numerical missing values:** Significant missingness exists in several property-related numerical variables, with some features having approximately **69% missing values**.

- **Extreme numerical observations:** `AMT_INCOME_TOTAL` contains a small number of extremely high income values, with a maximum of **117 million**. `CNT_CHILDREN` has a maximum of **19**, which is highly unusual but not automatically considered invalid.

- **Negative values:** Negative values are primarily present in `DAYS_*` variables and are valid because these features represent durations relative to the application date.

- **Special numerical value:** `DAYS_EMPLOYED` contains **55,374 observations with the value `365243`**, which represents a special/unknown employment status rather than a genuine employment duration. This value will require appropriate treatment during preprocessing.

- **Zero values:** Several numerical variables contain a very high proportion of zeros. This is particularly evident in the `FLAG_DOCUMENT_*` variables and is consistent with their document-indicator nature.

- **Constant features:** No completely constant numerical features were identified.

- **Near-constant features:** **16 `FLAG_DOCUMENT_*` variables** contain at least **99% zero values**. These are not considered data-quality errors but may have limited predictive value and will be evaluated during feature selection.

- **Data types:** The 16 `object` columns correspond to the identified categorical variables. No obvious data-type inconsistencies were identified.

### Overall Assessment

The dataset contains several important data-quality considerations, particularly **missing values, special values, rare categories, extreme numerical observations, and near-constant features**. However, no major structural data-quality issue was identified that would prevent further analysis.

At this stage, **no observations or features are being removed solely based on these findings**. The identified issues will be addressed using appropriate preprocessing and feature-engineering strategies after further exploratory analysis and consideration of their business and predictive significance.

### Data Quality Assessment Status

**Data Quality Assessment: Completed ✅**

# Step 3.1 — Data Dictionary Construction

In [35]:
print(application_train_csv.columns.tolist())

['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELE

In [36]:
data_dictionary=pd.DataFrame({'columns':application_train_csv.columns,
                              'data_type':application_train_csv.dtypes.astype(str).values})
data_dictionary

,columns,data_type
0,SK_ID_CURR,int64
1,TARGET,int64
2,NAME_CONTRACT_TYPE,object
3,CODE_GENDER,object
4,FLAG_OWN_CAR,object
...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,float64
118,AMT_REQ_CREDIT_BUREAU_WEEK,float64
119,AMT_REQ_CREDIT_BUREAU_MON,float64
120,AMT_REQ_CREDIT_BUREAU_QRT,float64


In [37]:
data_dictionary['missing_count']=application_train_csv.isnull().sum().values
data_dictionary['missing_percent']=(data_dictionary['missing_count']/len(application_train_csv)*100).round(2)
data_dictionary

,columns,data_type,missing_count,missing_percent
0,SK_ID_CURR,int64,0,0.00
1,TARGET,int64,0,0.00
2,NAME_CONTRACT_TYPE,object,0,0.00
3,CODE_GENDER,object,0,0.00
4,FLAG_OWN_CAR,object,0,0.00
...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,float64,41519,13.50
118,AMT_REQ_CREDIT_BUREAU_WEEK,float64,41519,13.50
119,AMT_REQ_CREDIT_BUREAU_MON,float64,41519,13.50
120,AMT_REQ_CREDIT_BUREAU_QRT,float64,41519,13.50


In [38]:
application_train_csv.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,"202,500.00","406,597.50","24,700.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,1.00
1,100003,0,Cash loans,F,N,N,0,"270,000.00","1,293,502.50","35,698.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00
2,100004,0,Revolving loans,M,Y,Y,0,"67,500.00","135,000.00","6,750.00",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00
3,100006,0,Cash loans,F,N,Y,0,"135,000.00","312,682.50","29,686.50",...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,"121,500.00","513,000.00","21,865.50",...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00


In [39]:
application_train_csv[['AMT_GOODS_PRICE','NAME_TYPE_SUITE']].head()

,AMT_GOODS_PRICE,NAME_TYPE_SUITE
0,"351,000.00",Unaccompanied
1,"1,129,500.00",Family
2,"135,000.00",Unaccompanied
3,"297,000.00",Unaccompanied
4,"513,000.00",Unaccompanied


In [40]:
application_train_csv[['OCCUPATION_TYPE','CNT_FAM_MEMBERS']].head(10)

,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,Laborers,1.00
1,Core staff,2.00
2,Laborers,1.00
3,Laborers,2.00
4,Core staff,1.00
5,Laborers,2.00
6,Accountants,3.00
7,Managers,2.00
8,NaN,2.00
9,Laborers,1.00


In [44]:
application_train_csv['TARGET'].sample(10)

10612     0
302652    1
230385    0
292413    0
109311    0
59014     0
214612    0
141996    0
196450    0
14774     0
Name: TARGET, dtype: int64

# Analysis of Target variable

In [45]:
application_train_csv['TARGET'].value_counts()

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [46]:
target_summary=pd.DataFrame({'count':application_train_csv['TARGET'].value_counts(),
'percentage':application_train_csv['TARGET'].value_counts(normalize=True)*100})
target_summary

,count,percentage
TARGET,,
0,282686,91.93
1,24825,8.07


In [47]:
application_train_csv['TARGET'].nunique()

2

In [48]:
application_train_csv['TARGET'].isnull().sum()

np.int64(0)

## Target Variable Exploration

The target variable `TARGET` represents whether an applicant experienced payment difficulties:

- `TARGET = 0` → Applicant did not experience payment difficulties.
- `TARGET = 1` → Applicant experienced payment difficulties.

The target variable contains **307,511 observations** with **2 unique values** and **no missing values**.

| TARGET | Count | Percentage |
|:------:|------:|-----------:|
| 0 | 282,686 | 91.93% |
| 1 | 24,825 | 8.07% |

The target variable is **highly imbalanced**, with only **8.07%** of applicants belonging to the positive class (`TARGET = 1`).

This class imbalance is important for the credit risk modeling process because a model that predicts most applicants as `TARGET = 0` could achieve high accuracy while failing to identify risky customers. Therefore, accuracy alone will not be sufficient for evaluating our models. Metrics such as **recall, precision, ROC-AUC, and PR-AUC** will be considered during the modeling phase.

From a business perspective, correctly identifying applicants with payment difficulties is particularly important because failing to identify a risky applicant may result in financial losses for the lending institution.